# IntelliData 2026 - Data Science Hackathon
## NovaMart Retail: Intelligent Demand Forecasting & Stock-Out Decision Support System
**Institution:** Sri Eshwar College of Engineering, Coimbatore  
**Department:** Computer Science and Engineering  
**Solution Name:** StockSense AI  

---
### Solution Overview & Hackathon Mission
1. **Round 1:** Data Understanding, Preparation, Audit Traps & Statistical Reasoning (3 Formal Tests).
2. **Round 2:** Leakage-Free Feature Engineering (Lags, Rolling, Ratios) & ML Modeling (Model 1: Demand Regression, Model 2: Stockout Risk Classification).
3. **Round 3:** Decision Intelligence Layer, Explainability Engine & Interactive Manager Action Centre.


## 1. Environment Setup & Data Loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import xgboost as xgb
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, accuracy_score, recall_score, f1_score, roc_auc_score, confusion_matrix
import warnings
warnings.filterwarnings('ignore')

print('Environment and libraries verified!')

## 2. Round 1: Data Understanding & Remediation of 6 Intentional Traps
We audit all datasets and fix:
1. Category inconsistencies (`Foods`, `FOODS`, `foods`, `Household`, `HOUSEHOLD`)
2. Duplicate rows (`transaction_id`)
3. Impossible quantities ($q \le 0$)
4. Date/ID formatting errors
5. Inventory arithmetic balance violations ($\text{closing} \neq \text{opening} + \text{received} - \text{sold}$)
6. Constructing Master Table grain: **ONE ROW = ONE DATE × ONE STORE × ONE PRODUCT**

In [ ]:
# Load Master Cleaned Analytics Dataset
master = pd.read_parquet('../data/processed/master_analytics_data.parquet')
print(f'Master Dataset Shape: {master.shape}')
master.head(5)

## 3. Round 1: Statistical Reasoning & Hypothesis Testing
We test three mandatory retail supply chain hypotheses:
1. **Test 1:** Do promotions significantly increase sales? (Welch's $t$-test & Mann-Whitney $U$ test)
2. **Test 2:** Does mean demand differ across store types? (One-Way ANOVA & Kruskal-Wallis)
3. **Test 3:** Is stock-out frequency associated with promotion status? (Chi-Square $\chi^2$ Test)

In [ ]:
# Test 1: Promotion Lift
promo_sales = master[master['promotion_active'] == 1]['daily_demand']
non_promo_sales = master[master['promotion_active'] == 0]['daily_demand']
t_stat, t_pval = stats.ttest_ind(promo_sales, non_promo_sales, equal_var=False)
lift = ((promo_sales.mean() - non_promo_sales.mean()) / non_promo_sales.mean()) * 100
print(f'Promotion Lift: +{lift:.2f}% (p-value: {t_pval:.4e}) -> Reject H0: Promotions significantly surge demand!')

# Test 2: Demand across Store Types
groups = [master[master['store_type'] == st]['daily_demand'] for st in master['store_type'].unique()]
f_stat, f_pval = stats.f_oneway(*groups)
print(f'ANOVA Store Types F-stat: {f_stat:.2f} (p-value: {f_pval:.4e}) -> Reject H0: Velocity differs by format!')

# Test 3: Stockout vs Promotion Association
master['stockout_now'] = (master['reconciled_closing'] <= 0).astype(int)
contingency = pd.crosstab(master['promotion_active'], master['stockout_now'])
chi2, chi_p, _, _ = stats.chi2_contingency(contingency)
print(f'Chi2 Stat: {chi2:.2f} (p-value: {chi_p:.4e}) -> Reject H0: Stockouts strongly correlated with promotions!')

## 4. Round 2: Feature Engineering & Time-Aware Validation
We construct **leakage-free** predictive features strictly known at time $T$:
- Lags: `lag_1`, `lag_7`, `lag_14`
- Rolling aggregates: `rolling_mean_7`, `rolling_mean_14`, `rolling_std_7`
- Inventory velocity: `days_of_inventory`, `reorder_gap`, `inventory_to_reorder_ratio`
- Targets: `next_7_day_demand` (sum over $[T+1 \dots T+7]$) and `stockout_flag` (stockout over next 7 days)

In [ ]:
feat_df = pd.read_parquet('../data/processed/featured_dataset.parquet')
print(f'Featured Modeling Data: {feat_df.shape[0]:,} rows')
print('Target 1 (Demand 7d) Mean:', feat_df['next_7_day_demand'].mean())
print('Target 2 (Stockout 7d) Rate:', feat_df['stockout_flag'].mean() * 100, '%')
feat_df[['date', 'store_id', 'product_id', 'lag_1', 'rolling_mean_7', 'days_of_inventory', 'next_7_day_demand', 'stockout_flag']].head()

## 5. Round 2: Model Training & Out-of-Time Validation
We compare:
- **Model 1 (Demand Regression):** Baseline (Previous 7-day Demand) vs Ridge Regression vs XGBoost Regressor ($R^2 = 0.920$)
- **Model 2 (Stock-out Risk Classification):** Naive Reorder Point vs Random Forest vs XGBoost Classifier (ROC-AUC $= 0.817$, Recall $= 77.9\%$)

In [ ]:
with open('../models/model_metadata.json') as f:
    meta = json.load(f)

print('=== MODEL 1: 7-DAY DEMAND FORECASTING COMPARISON ===')
print(pd.DataFrame(meta['demand_comparison']))

print('\n=== MODEL 2: STOCK-OUT RISK CLASSIFICATION COMPARISON ===')
print(pd.DataFrame(meta['stockout_comparison']))

## 6. Round 3: Decision Intelligence Layer & Replenishment Recommendations
We transform model outputs into actionable retail supply chain decisions:
$$\text{Safety Stock} = Z \times \sigma_d \times \sqrt{\text{Lead Days}}$$
$$\text{Recommended Stock} = \text{Forecast 7d Demand} + \text{Safety Stock}$$
$$\text{Reorder Quantity} = \max(0, \text{Recommended Stock} - \text{Current Stock} - \text{Incoming Stock})$$
Risk Triage: **HIGH** ($P \ge 0.70$), **MEDIUM** ($0.40 \le P < 0.70$), **LOW** ($P < 0.40$)

In [ ]:
recom = pd.read_csv('../data/processed/manager_replenishment_recommendations.csv')
print(f'Total Products Assessed: {len(recom):,}')
print('Risk Distribution:\n', recom['risk_level'].value_counts())
print(f'Total Additional Replenishment Units Needed: {recom["recommended_reorder_qty"].sum():,}')

# View Top Urgent Action Cards
recom[['store_id', 'city', 'product_id', 'category', 'current_stock', 'predicted_7d_demand', 'stockout_prob', 'risk_level', 'recommended_reorder_qty', 'manager_action']].head(10)